# Wearwell 생성형 이미지 API — GPU 런타임

이 노트북은 **FLUX.2 [klein] 4B** 하나로 체형 아바타 생성과 여러 의류의 다중 참조 가상 착장을 처리합니다. 모델 API와 터널만 실행하며 프론트엔드 파일은 로컬에서 제공합니다.

> `런타임 → 런타임 유형 변경`에서 L4, A100 또는 H100 GPU를 선택하세요. L4 24GB로 실행 가능하고 A100/H100에서는 더 여유롭게 동작합니다.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if sys.version_info < (3, 10):
    raise RuntimeError(f"Python 3.10+가 필요합니다. 현재: {sys.version}")

print("Python:", sys.version.split()[0])
subprocess.run(["nvidia-smi"], check=True)

try:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("GPU 런타임이 아닙니다. 런타임 유형에서 GPU를 선택하세요.")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name} ({gpu_memory_gb:.1f} GB)")
    if gpu_memory_gb < 16:
        raise RuntimeError("FLUX.2 [klein]에는 VRAM 16GB 이상을 권장합니다.")
except ImportError as exc:
    raise RuntimeError("GPU 런타임의 PyTorch를 찾지 못했습니다.") from exc

In [ ]:
import shutil

REPO_URL = "https://github.com/dongwgo/wearwell.git"
REPO_DIR = Path("/content/wearwell")
HF_HOME = Path("/content/hf-cache")
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

def run(*args, cwd=None):
    print("+", " ".join(map(str, args)))
    subprocess.run(list(map(str, args)), cwd=cwd, check=True)

if (REPO_DIR / ".git").exists():
    run("git", "pull", "--ff-only", cwd=REPO_DIR)
else:
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    run("git", "clone", "--depth", "1", REPO_URL, REPO_DIR)

run(sys.executable, "-m", "pip", "install", "-q", "hf_transfer>=0.1.9")
run(sys.executable, "-m", "pip", "install", "-q", "-r", REPO_DIR / "backend/requirements.txt")
print("백엔드와 FLUX.2 파이프라인 설치 완료")

In [ ]:
import json
import secrets
import time
import urllib.request

BACKEND_PORT = 8787
API_TOKEN = secrets.token_urlsafe(32)
SERVER_LOG = Path("/content/wearwell-uvicorn.log")

for name in ("wearwell_server", "wearwell_tunnel"):
    process = globals().get(name)
    if process and process.poll() is None:
        process.terminate()
        process.wait(timeout=10)

env = os.environ.copy()
env.update({
    "HF_HOME": str(HF_HOME),
    "HF_HUB_ENABLE_HF_TRANSFER": "1",
    "IMAGE_MODEL": "black-forest-labs/FLUX.2-klein-4B",
    "IMAGE_WIDTH": "768",
    "IMAGE_HEIGHT": "1152",
    "FLUX_STEPS": "4",
    "FLUX_GUIDANCE": "1.0",
    "FLUX_CPU_OFFLOAD": "0" if gpu_memory_gb >= 20 else "1",
    "ONEULOUT_GPU": "1",
    "WEARWELL_API_TOKEN": API_TOKEN,
    "PYTHONUNBUFFERED": "1",
})

server_log_handle = SERVER_LOG.open("w")
wearwell_server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", str(BACKEND_PORT)],
    cwd=REPO_DIR / "backend",
    env=env,
    stdout=server_log_handle,
    stderr=subprocess.STDOUT,
)

health_url = f"http://127.0.0.1:{BACKEND_PORT}/api/health"
for _ in range(90):
    if wearwell_server.poll() is not None:
        raise RuntimeError(SERVER_LOG.read_text(errors="replace"))
    try:
        with urllib.request.urlopen(health_url, timeout=2) as response:
            health = json.load(response)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("백엔드 시작 시간 초과\n" + SERVER_LOG.read_text(errors="replace"))

warmup_request = urllib.request.Request(
    f"http://127.0.0.1:{BACKEND_PORT}/api/warmup",
    method="POST",
    headers={"Authorization": f"Bearer {API_TOKEN}"},
)
print("FLUX.2 모델 다운로드·로드 중입니다. 첫 실행은 수 분 걸릴 수 있습니다.")
with urllib.request.urlopen(warmup_request, timeout=2400) as response:
    print(json.dumps(json.load(response), ensure_ascii=False, indent=2))
with urllib.request.urlopen(health_url, timeout=5) as response:
    health = json.load(response)
if not health.get("warmupVerified"):
    raise RuntimeError("Model warmup verification failed")
print(json.dumps(health, ensure_ascii=False, indent=2))

In [ ]:
import hashlib
import re

CLOUDFLARED = Path("/content/cloudflared")
CLOUDFLARED_VERSION = "2026.8.2"
CLOUDFLARED_SHA256 = "fcfb02b575a52ca1af2e3267af4e1517bcdeb30ac48c834c69abaed3c0576ad2"
TUNNEL_LOG = Path("/content/cloudflared.log")
if CLOUDFLARED.exists() and hashlib.sha256(CLOUDFLARED.read_bytes()).hexdigest() != CLOUDFLARED_SHA256:
    CLOUDFLARED.unlink()
if not CLOUDFLARED.exists():
    urllib.request.urlretrieve(
        f"https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-amd64",
        CLOUDFLARED,
    )
    if hashlib.sha256(CLOUDFLARED.read_bytes()).hexdigest() != CLOUDFLARED_SHA256:
        CLOUDFLARED.unlink(missing_ok=True)
        raise RuntimeError("cloudflared checksum mismatch")
CLOUDFLARED.chmod(0o755)

tunnel_log_handle = TUNNEL_LOG.open("w")
wearwell_tunnel = subprocess.Popen(
    [str(CLOUDFLARED), "tunnel", "--url", f"http://127.0.0.1:{BACKEND_PORT}", "--no-autoupdate"],
    stdout=tunnel_log_handle,
    stderr=subprocess.STDOUT,
)

backend_url = None
for _ in range(90):
    if wearwell_tunnel.poll() is not None:
        raise RuntimeError(TUNNEL_LOG.read_text(errors="replace"))
    text = TUNNEL_LOG.read_text(errors="replace") if TUNNEL_LOG.exists() else ""
    match = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", text)
    if match:
        backend_url = match.group(0)
        break
    time.sleep(1)
if not backend_url:
    raise RuntimeError("Cloudflare Tunnel URL 생성 시간 초과\n" + TUNNEL_LOG.read_text(errors="replace"))

local_config_path = Path("/content/local-config.js")
local_config_path.write_text(
    "window.WEARWELL_CONFIG = "
    + json.dumps({"API_BASE": backend_url, "API_TOKEN": API_TOKEN}, ensure_ascii=False)
    + ";\n",
    encoding="utf-8",
)
print("BACKEND_URL=" + backend_url)
print("모델 API가 준비되었습니다.")

from google.colab import files
files.download(str(local_config_path))

## 로컬 프론트엔드 연결

1. 마지막 셀에서 받은 `local-config.js`를 Wearwell 프로젝트 루트에 둡니다.
2. 프로젝트 루트에서 `python -m http.server 8000 --bind 127.0.0.1`을 실행합니다.
3. 브라우저에서 `http://127.0.0.1:8000`을 엽니다.

`local-config.js`에는 세션 인증 토큰이 있으므로 Git에 추가하거나 공유하지 마세요. 런타임을 재시작하면 마지막 셀에서 새 파일을 다시 받으세요.